# Blender GUI + Chrome Remote Desktop on Google Colab | v1.0.4
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tbcreator71/Google-Colab-Notebooks/blob/Blender_Notebooks/Blender_GUI_Chrome_Remote_Desktop.ipynb)

**Purpose:** Test Blender add-ons (including Instant Hair) remotely from any browser. Blender and its CUDA dependencies run on Colab. Your PC needs no NVIDIA GPU.

**Default accelerator:** NVIDIA T4 GPU (16 GB). Colab may override a notebook preference based on availability; confirm Runtime → Change runtime type → T4 GPU.\n\n**Workflow:** 1. Settings & Drive → 2. Desktop packages → 3. Blender install → 4. Chrome Remote Desktop authorization → 5. Open remote desktop & launch Blender → 6. Optional add-on checkout → 7. Diagnostics / backup.

**Caching:** Blender archive, optional add-on repository, and saved Blender preferences/projects persist in Google Drive. Colab VM system packages, the remote desktop host, and Python environment must be reinstalled every *new* runtime. Blender is extracted to local disk for speed. This does not promise instant startup.

**Important requirements and limitations**
- Use **paid Colab with positive compute units**. Google's rules restrict remote desktops on managed free-tier runtimes: https://research.google.com/colaboratory/faq.html
- The Colab GPU is available for CUDA inference, but **Chrome Remote Desktop does not provide GPU-accelerated Blender viewport**. Mesa software OpenGL may work but is not guaranteed.
- All runtimes are temporary. Do not leave unsaved work only on the remote desktop.
- Never put Google authorization codes, your PIN, repository tokens, or model credentials into the notebook source.
- Default Blender **4.5.16 LTS**. Change version in settings for a compatible version. Verify compatibility of your add-on separately.
- No models are automatically downloaded. For gated SAM3/DINOv3 weights, observe licensing and use approved private access methods.


In [ ]:
# 1. Settings, persistent Drive cache, runtime information
from google.colab import drive
from pathlib import Path
import os, subprocess, shutil, json, time
drive.mount("/content/drive")

BLENDER_VERSION = "4.5.16"  # @param {type:"string"}
CACHE_ROOT = Path("/content/drive/MyDrive/Blender-Colab-Cache") # @param {type:"string"}
LOCAL_ROOT = Path("/content/blender-colab")
DESKTOP_USER = "blenderuser"
INSTALL_DIR = LOCAL_ROOT / "software"
BLENDER_DIR = INSTALL_DIR / f"blender-{BLENDER_VERSION}-linux-x64"
ARCHIVE = CACHE_ROOT / "downloads" / f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
PROJECT_DIR = CACHE_ROOT / "projects"
ADDON_CACHE = CACHE_ROOT / "addons"
BACKUP_DIR = CACHE_ROOT / "backups"
for p in [CACHE_ROOT, ARCHIVE.parent, INSTALL_DIR, PROJECT_DIR, ADDON_CACHE, BACKUP_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print("Preferred GPU: NVIDIA T4 (16 GB). Verify Runtime → Change runtime type → T4 GPU.")
print("Blender version:", BLENDER_VERSION)
print("Archive cache:", ARCHIVE)
print("Local installation:", BLENDER_DIR)
print("Persistent projects:", PROJECT_DIR)
subprocess.run(["nvidia-smi"], check=False)
print("Disk free (GB):", round(shutil.disk_usage("/content").free/1e9, 1))


In [ ]:
# 2. Install virtual desktop, software OpenGL and Chrome Remote Desktop
# Re-run on each newly assigned Colab VM. apt packages are ephemeral.
import subprocess, pathlib
def run(cmd, **kwargs):
    print("+", " ".join(map(str,cmd)))
    subprocess.run(list(map(str,cmd)), check=True, **kwargs)

run(["apt-get","update","-qq"])
run(["apt-get","install","-y","-qq","xfce4","xfce4-terminal","dbus-x11","xauth",
     "mesa-utils","libgl1-mesa-dri","libegl1","libxrender1","libxi6",
     "libxkbcommon-x11-0","libsm6","libxfixes3","libxrandr2","libxcursor1",
     "libxinerama1","libxxf86vm1","libglib2.0-0","libfontconfig1","libfreetype6",
     "wget","curl","ca-certificates"])
crd_deb = pathlib.Path("/tmp/chrome-remote-desktop_current_amd64.deb")
run(["wget","-q","-O",str(crd_deb),"https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb"])
run(["apt-get","install","-y",str(crd_deb)])
if subprocess.run(["id","-u",DESKTOP_USER],capture_output=True).returncode != 0:
    run(["useradd","-m","-s","/bin/bash",DESKTOP_USER])
session=pathlib.Path(f"/home/{DESKTOP_USER}/.chrome-remote-desktop-session")
session.write_text("exec /etc/X11/Xsession /usr/bin/xfce4-session\n")
run(["chown",f"{DESKTOP_USER}:{DESKTOP_USER}",str(session)])
run(["usermod","-aG","video,render",DESKTOP_USER])
# Chrome Remote Desktop's host Python must match Ubuntu's compiled D-Bus bindings.
# Colab uses Python 3.13 while Ubuntu 24.04's python3-dbus is built for Python 3.12.
# Do not change /usr/bin/python3: that can break other Colab tools.
def verify_dbus():
    import re, shutil
    extensions=list(pathlib.Path("/usr/lib/python3/dist-packages").glob("_dbus_bindings.cpython-*-*.so"))
    if not extensions:
        raise RuntimeError("No native D-Bus binding found. Check python3-dbus installation.")
    versions={m.group(1) for p in extensions if (m:=re.search(r"cpython-(\d+)-",p.name))}
    if len(versions)!=1:
        raise RuntimeError("Ambiguous D-Bus ABI versions: "+str(versions))
    digits=versions.pop()
    version=digits[0]+"."+digits[1:]
    interpreter=pathlib.Path("/usr/bin/python"+version)
    if not interpreter.is_file():
        # Installing the matching distro interpreter does not change Colab's default.
        run(["apt-get","install","-y","python"+version])
    if not interpreter.is_file():
        raise RuntimeError("Missing required Python "+version)
    probe=subprocess.run([str(interpreter),"-c",
        "import sys, dbus, _dbus_bindings; print(sys.executable, sys.version.split()[0], _dbus_bindings.__file__)"],
        capture_output=True,text=True)
    if probe.returncode:
        raise RuntimeError("Python "+version+" D-Bus import failed:\n"+probe.stderr[-1600:])
    print("D-Bus verified:",probe.stdout.strip())
    host_script=pathlib.Path("/opt/google/chrome-remote-desktop/chrome-remote-desktop")
    if not host_script.is_file():
        raise RuntimeError("Chrome Remote Desktop host script not found: "+str(host_script))
    data=host_script.read_bytes()
    old_line=data.split(b"\n",1)[0]
    if not old_line.startswith(b"#!") or b"python3" not in old_line:
        raise RuntimeError("Unexpected CRD host script interpreter: "+repr(old_line))
    new_line=("#!"+str(interpreter)).encode()
    if old_line!=new_line:
        # Modify only the interpreter line, not the globally installed Python.
        host_script.with_name(host_script.name+".colab-original").write_bytes(data)
        host_script.write_bytes(new_line+data[len(old_line):])
        host_script.chmod(0o755)
        print("Set CRD host script interpreter to",interpreter)
    check=subprocess.run([str(host_script),"--help"],text=True,capture_output=True,timeout=30)
    # --help may not be a supported option; only check for Python import failure.
    if "_dbus_bindings" in (check.stdout+check.stderr) and "ModuleNotFoundError" in (check.stdout+check.stderr):
        raise RuntimeError("CRD host still cannot import D-Bus: "+check.stderr[-900:])
    print("CRD host Python preflight completed. The setup may now proceed.")
verify_dbus()
print("Desktop and Chrome Remote Desktop installed.")


In [ ]:
# 3. Restore/download Blender into fast local VM disk
# Uses the exact release archive, cached on Drive between sessions.
import urllib.request, tarfile, os, shutil, subprocess
from pathlib import Path
url = f"https://download.blender.org/release/Blender{'.'.join(BLENDER_VERSION.split('.')[:2])}/blender-{BLENDER_VERSION}-linux-x64.tar.xz"
if not ARCHIVE.is_file() or ARCHIVE.stat().st_size < 100_000_000:
    tmp = ARCHIVE.with_suffix(".partial")
    print("Downloading official Blender build:", url)
    try:
        urllib.request.urlretrieve(url, tmp)
        if tmp.stat().st_size < 100_000_000: raise RuntimeError("Downloaded archive unexpectedly small")
        tmp.replace(ARCHIVE)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise
else:
    print("Using cached Blender archive", ARCHIVE)
if not (BLENDER_DIR/"blender").exists():
    # Pre-validate archive and avoid extraction of links outside the destination.
    with tarfile.open(ARCHIVE,"r:xz") as tar:
        root=INSTALL_DIR.resolve()
        for item in tar.getmembers():
            target=(INSTALL_DIR/item.name).resolve()
            if not target.is_relative_to(root):
                raise RuntimeError("Unsafe archive member: "+item.name)
        tar.extractall(INSTALL_DIR,filter="data")
    print("Extracted",BLENDER_DIR)
else:
    print("Blender already extracted")
binpath=BLENDER_DIR/"blender"
assert binpath.is_file(),f"Blender executable missing: {binpath}"
subprocess.run([str(binpath),"-b","--factory-startup","--python-expr","import bpy; print('BLENDER_OK',bpy.app.version_string)"],check=True,timeout=100)
# Enable desktop user to read application files, and provide convenient launcher.
subprocess.run(["chmod","-R","a+rX",str(INSTALL_DIR)],check=True)
launcher=Path("/usr/local/bin/blender-colab")
launcher.write_text("#!/bin/sh\nexport LIBGL_ALWAYS_SOFTWARE=1\nexport GALLIUM_DRIVER=llvmpipe\nexec '"+str(binpath)+"' \"$@\"\n")
launcher.chmod(0o755)
desktop=Path(f"/home/{DESKTOP_USER}/Desktop")
desktop.mkdir(exist_ok=True)
link=desktop/"Blender.desktop"
link.write_text("[Desktop Entry]\nType=Application\nName=Blender (Colab)\nExec=blender-colab\nTerminal=false\nCategories=Graphics;\n")
link.chmod(0o755)
subprocess.run(["chown","-R",f"{DESKTOP_USER}:{DESKTOP_USER}",str(desktop)],check=True)
print("Ready. Run blender-colab in your remote XFCE terminal.")


## 4. Register the temporary Colab machine in Chrome Remote Desktop

1. On your own computer open **https://remotedesktop.google.com/headless** and sign in.
2. Select the Debian/Linux setup and copy the full **DISPLAY= /opt/google/chrome-remote-desktop/start-host ...** command.
3. Execute the cell below. Paste the command into its hidden input, then enter a new numeric PIN (6+ digits).
4. Open **https://remotedesktop.google.com/access**, select the newly registered **Colab-Blender** host, and enter your PIN.

**Security:** Treat the authorization command like a temporary credential. This notebook does not persist it. Each new Colab VM normally needs new authorization. If the host entry remains offline after a VM reset, remove the stale device from Chrome Remote Desktop before registering the new one.


In [ ]:
# 4. Register Chrome Remote Desktop with safe diagnostics (v1.0.4)
# Get a NEW single-use command at https://remotedesktop.google.com/headless
import getpass, shlex, subprocess, re, os, socket
# Validate dbus before consuming a one-time Google registration code.
verify_dbus()
raw = getpass.getpass("Paste full DISPLAY= ... start-host command: ").strip()
parts = shlex.split(raw)
if parts and parts[0] == "DISPLAY=":
    parts.pop(0)
if not parts or parts[0] != "/opt/google/chrome-remote-desktop/start-host":
    raise ValueError("Paste the Debian/Linux command from the official Chrome Remote Desktop headless page.")
options = parts[1:]
if not any(x.startswith("--code=") for x in options):
    raise ValueError("Missing --code. Generate a new Linux authorization command.")
if not any(x.startswith("--redirect-url=") for x in options):
    raise ValueError("Missing --redirect-url.")
if not all(x.startswith(("--code=", "--redirect-url=", "--name=")) for x in options):
    raise ValueError("Unexpected command options. Copy the official headless command without modifications.")
# The official command uses shell substitution; subprocess does not expand $(hostname).
options = [("--name=Colab-Blender" if x == "--name=$(hostname)" else x) for x in options]
if not any(x.startswith("--name=") for x in options):
    options.append("--name=Colab-Blender")
pin = getpass.getpass("New numeric PIN (at least 6 digits): ")
if not (pin.isdigit() and len(pin) >= 6):
    raise ValueError("The PIN must contain at least six digits.")
env = dict(os.environ)
env["DISPLAY"] = ""
# Execute as the desktop user, not as root.
cmd = ["runuser", "-u", DESKTOP_USER, "--", parts[0], *options, "--pin=" + pin]
secrets = [pin] + [x.partition("=")[2] for x in options if x.startswith("--code=")]
try:
    p = subprocess.run(cmd, env=env, text=True, capture_output=True, timeout=120)
    # Redact credentials before printing any diagnostics; also mask possible OAuth codes in logs.
    diagnostic = (p.stdout + "\n" + p.stderr)
    for token in secrets:
        if token:
            diagnostic = diagnostic.replace(token, "[REDACTED]")
    diagnostic = re.sub(r'(?i)(--code[=\s]+)[^\s]+', r'\1[REDACTED]', diagnostic)
    diagnostic = re.sub(r'(?i)(--pin[=\s]+)[^\s]+', r'\1[REDACTED]', diagnostic)
    diagnostic = re.sub(r'4/[A-Za-z0-9_\-]{16,}', '[REDACTED]', diagnostic)
    diagnostic = diagnostic[-4500:]
    print("Host registration exit code:", p.returncode)
    if p.returncode:
        print("Sanitized Chrome Remote Desktop output:\n", diagnostic.strip() or "(No stdout/stderr returned)")
        print("For an expired or already used code, generate a new command at https://remotedesktop.google.com/headless")
    else:
        print("Registered. Visit https://remotedesktop.google.com/access to connect.")
finally:
    raw = ""; parts = []; options = []; cmd = []; pin = ""; secrets = []


## 5. Launch Blender from Chrome Remote Desktop

In the remote XFCE desktop, open **Applications → Terminal Emulator** and run:

```bash
blender-colab
```

Blender is configured to try **Mesa llvmpipe** software OpenGL for the viewport. This may be slower than a local GPU and may fail depending on the runtime's supported OpenGL extensions. CUDA acceleration for external Python/HairDeck processes is separate from the viewport.

**Important:** Don't assume GPU inference dependencies are installed just because Blender starts. Install add-on requirements according to its repository and check for conflicting Blender-bundled Python versus Colab Python packages.


In [ ]:
# 6. Optional: cache an add-on Git repository and make it available locally
# Supply a public HTTPS URL. For private repositories use a separate authenticated workflow;
# do not place access tokens in notebook source.
ADDON_REPO_URL = ""  # @param {type:"string"}
ADDON_REPO_REF = "main" # @param {type:"string"}
import subprocess, re, shutil
from pathlib import Path
if ADDON_REPO_URL.strip():
    if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?",ADDON_REPO_URL.strip()):
        raise ValueError("Use a plain public GitHub HTTPS repository URL without credentials")
    repo_name=ADDON_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    local=LOCAL_ROOT/"addons"/repo_name
    local.parent.mkdir(parents=True,exist_ok=True)
    mirror=ADDON_CACHE/(repo_name+".git")
    if not mirror.exists():
        subprocess.run(["git","clone","--mirror",ADDON_REPO_URL,str(mirror)],check=True)
    else:
        subprocess.run(["git","--git-dir",str(mirror),"remote","update","--prune"],check=True)
    if not local.exists():
        subprocess.run(["git","clone",str(mirror),str(local)],check=True)
    subprocess.run(["git","-C",str(local),"fetch",str(mirror),ADDON_REPO_REF],check=True)
    subprocess.run(["git","-C",str(local),"checkout","--detach","FETCH_HEAD"],check=True)
    print("Add-on sources:",local)
    print("To install, use Blender Preferences → Add-ons → Install from Disk (ZIP), or install an appropriate packaged Blender extension.")
else:
    print("No add-on repo configured. Install the add-on from the Blender GUI or set URL and re-run.")


In [ ]:
# 7. Diagnostics: GPU, renderer, filesystem and useful logs
import subprocess, os
print("=== CUDA ===")
subprocess.run(["nvidia-smi"],check=False)
print("=== Blender binary ===", binpath.exists())
print("=== Directories ===")
print("Cache", CACHE_ROOT, "exists", CACHE_ROOT.exists())
print("Local install",BLENDER_DIR,"exists",BLENDER_DIR.exists())
print("=== Chrome Remote Desktop host ===")
subprocess.run(["pgrep","-af","chrome-remote-desktop"],check=False)
print("=== GUI OpenGL check ===")
print("In remote XFCE Terminal run: LIBGL_ALWAYS_SOFTWARE=1 GALLIUM_DRIVER=llvmpipe glxinfo -B")
print("For Blender logs run: blender-colab --debug")


In [ ]:
# 8. Save / restore Blender user settings and projects
# Run this AFTER modifying Blender settings, with Blender closed if possible.
# GUI user home is not persistent. This saves only user settings, not full OS environment.
import tarfile, shutil
from pathlib import Path
config=Path(f"/home/{DESKTOP_USER}/.config/blender")
backup=BACKUP_DIR/"blender-user-settings.tar.gz"
def save_blender_settings():
    if not config.exists():
        print("No Blender user settings yet; launch Blender first.")
        return
    with tarfile.open(backup,"w:gz") as tar:
        tar.add(config,arcname="blender")
    print("Saved Blender settings to",backup)
def restore_blender_settings():
    if not backup.exists():
        print("No settings backup yet")
        return
    config.parent.mkdir(parents=True,exist_ok=True)
    with tarfile.open(backup,"r:gz") as tar:
        tar.extractall(config.parent,filter="data")
    subprocess.run(["chown","-R",f"{DESKTOP_USER}:{DESKTOP_USER}",str(config)],check=True)
    print("Restored settings")
ACTION = "restore" # @param ["restore", "save"]
if ACTION=="save": save_blender_settings()
else: restore_blender_settings()
print("Persistent projects folder:",PROJECT_DIR)
print("NOTE: Blender remote GUI may not have write access to mounted Drive. Save to your local home then copy projects through this notebook.")


## Troubleshooting and maintenance

- **Blender doesn't start:** From the remote terminal run `LIBGL_ALWAYS_SOFTWARE=1 GALLIUM_DRIVER=llvmpipe blender-colab --debug`; check `glxinfo -B` for OpenGL >=4.3. A100 CUDA visibility alone does not guarantee usable graphical OpenGL.
- **Remote host offline:** Colab may have recycled the VM; rerun setup and register a fresh Chrome Remote Desktop host.
- **Crash or blank window:** Software GL may not implement all required driver features; use Blender background mode for API tests or a GPU workstation with graphics-capable drivers.
- **Permission errors:** Remote desktop user is `blenderuser`, notebook user is root. Project saves under `/home/blenderuser` are ephemeral; copy finished work to Drive.
- **Cache behavior:** Drive caches archive bytes, not apt-installed system packages. On a new VM Blender is still extracted locally before use.
- **Security:** Don't expose a public unauthenticated remote desktop. Store any gated-model token in Colab Secrets rather than inside Git history.

- **`ModuleNotFoundError: No module named '_dbus_bindings'`:** CRD imports Ubuntu's binary D-Bus Python bindings; Colab may have a mismatched Python interpreter. Re-run installation cell, which detects the D-Bus extension ABI, validates matching Ubuntu Python (typically 3.12), and updates *only* the Chrome Remote Desktop host script interpreter. If this fails, report diagnostics; do **not** change the global `/usr/bin/python3` symlink.
